In [69]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
import matplotlib.pyplot as plt


In [70]:
df = pd.read_pickle('../data/plot_features.pkl')

In [132]:
print(f'area_m2 mean:{np.mean(df['area_m2'])}')
print(f'area_m2 std:{np.std(df['area_m2'])}')
print(f'area_m2 min:{np.min(df['area_m2'])}')
print(f'area_m2 max:{np.max(df['area_m2'])}')

area_m2 mean:48626.460279797684
area_m2 std:30777.08071697935
area_m2 min:3847.4136081237716
area_m2 max:143733.26263731995


In [104]:
features = [
    # 'curve_mean',
    'pro_curve_mean',
    'plan_curve_mean',
    # 'elev_min',
    # 'elev_max',
    'elev_mean',
    # 'elev_dev_min',
    # 'elev_dev_max',
    # 'elev_dev_mean',

    'area_ha',
    'sandtotal_r',
    'silttotal_r',
    'claytotal_r',
    'awc_r',
    'cec7_r',
    'om_r',
    'ph1to1h2o_r',
    'ec_r',
    # 'profile_depth',
    # 'max_depth',
    'frag3to10_r',
    'fraggt10_r',
    'dbovendry_r',
    'caco3_r',
    'aspect_mean_sin',
    'aspect_mean_cos',

    'slope_x',
    'slope_y',
]

In [115]:
targets = [
    # 'mean_vigor',
    'mean_stability',
]

In [116]:
X = df[features].copy()
y = df[targets].copy()

In [117]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 284329)

In [118]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [119]:

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Check R² on training and test sets
print("Train R²:", lr.score(X_train_scaled, y_train))
print("Test R²:", lr.score(X_test_scaled, y_test))


Train R²: 0.3719123922959193
Test R²: 0.40723821557142625


In [121]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=kf, scoring="r2")
print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())


CV R² scores: [-2.1990584  -2.34201687  0.56787266 -0.64641412 -0.18822855]
Mean CV R²: -0.961569055773191


In [126]:
# Ridge with built-in cross-validation
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(X_train_scaled, y_train)
print("Ridge R² (train):", ridge.score(X_train_scaled, y_train))
print("Ridge coefficients:", ridge.coef_)

# Lasso with cross-validation
lasso = LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
print("Lasso R² (train):", lasso.score(X_train_scaled, y_train))
print("Lasso coefficients:", lasso.coef_)


Ridge R² (train): 0.3145228620216489
Ridge coefficients: [-0.0008212   0.00059427  0.00099455  0.00053837 -0.0005012   0.00125534
  0.00023432  0.00076671 -0.0010098   0.00040568  0.00143563  0.00023432
  0.          0.          0.00121992  0.00023432 -0.00119841  0.00119586
  0.00105925 -0.00121296]
Lasso R² (train): 0.2814399351183673
Lasso coefficients: [-0.          0.          0.          0.         -0.          0.00092295
  0.          0.         -0.          0.          0.0049231   0.
  0.          0.          0.          0.         -0.0005316   0.00138046
  0.         -0.00154028]


/home/simonhans/anaconda3/envs/GrapeExpectationsML/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1714: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
